In [8]:
import os
import pickle
from skimage.io import imread
from skimage.transform import resize
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score


# prepare data
input_dir = r'C:\Users\Admin\Music\Python\Web-Scraping_Web-Crawling\Scraper\Mecsu\Data_model\Data_Plier'
categories = [
    'crimping', 'cutting', 'flat_nose', 'hole_punch',
    'lineman', 'locking', 'needle_nose', 'round_nose',
    'tongue_groove', 'wire_stripper'
]


data = []
labels = []
for category_idx, category in enumerate(categories):
    for file in os.listdir(os.path.join(input_dir, category)):
        img_path = os.path.join(input_dir, category, file)
        img = imread(img_path)
        img = resize(img, (15, 15))
        data.append(img.flatten())
        labels.append(category_idx)

data = np.asarray(data)
labels = np.asarray(labels)

# train / test split
x_train, x_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, shuffle=True, stratify=labels)

# train classifier
classifier = SVC()

parameters = [{'gamma': [0.01, 0.001, 0.0001], 'C': [1, 10, 100, 1000]}]

grid_search = GridSearchCV(classifier, parameters)

grid_search.fit(x_train, y_train)

# test performance
best_estimator = grid_search.best_estimator_

y_prediction = best_estimator.predict(x_test)

score = accuracy_score(y_prediction, y_test)

print('{}% of samples were correctly classified'.format(str(score * 100)))

pickle.dump(best_estimator, open('./model.p', 'wb'))
label_map = {i: category for i, category in enumerate(categories)}
pickle.dump(label_map, open('./label_map.p', 'wb'))


39.375% of samples were correctly classified


In [ ]:
import pickle
from skimage.io import imread
from skimage.transform import resize
import numpy as np

# Load model và label map
model = pickle.load(open('./model.p', 'rb'))
label_map = pickle.load(open('./label_map.p', 'rb'))  # {0: 'crimping', 1: 'cutting', ...}

# Dự đoán ảnh mới
img_path = r'C:\Users\Admin\Desktop\tải xuống.jpg'
img = imread(img_path)
img = resize(img, (15, 15))  
img_flat = img.flatten().reshape(1, -1)

prediction = model.predict(img_flat)[0]
class_name = label_map[prediction]

print(f'Ảnh được dự đoán là: {class_name}')


Ảnh được dự đoán là: cutting
